In [ ]:
!git clone https://github.com/ml-matthew-lam/fashion_classification.git

Cloning into 'fashion_classification'...
remote: Enumerating objects: 27, done.
remote: Total 27 (delta 0), reused 0 (delta 0), pack-reused 27 (from 2)
Receiving objects: 100% (27/27), 58.94 MiB | 13.82 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [ ]:
import os
os.chdir('fashion_classification')

In [ ]:
from model import FashionModel

In [ ]:
import torch
import torch.nn as nn
from model import FashionModel
from dataset import train_loader, test_loader

model = FashionModel() # create an instance of FashionModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # check if T4 GPU runtime is available and choose it if so, otherwise, use CPU
model = model.to(device) # send model onto GPU chip
print(f"running on: {device}")

running on: cuda


We will use cross-entropy loss: $\displaystyle L = -\sum_{i=0}^{n-1} y_i \log p_i$,
where
*   $n$ is the number of classes
*   $y_i= 1$ if is the correct class is the $i$th class, zero otherwise
*   $p_i$ is the predicted probability of the image being in the $i$th class.

This simplifies to $\displaystyle L = -\log p_c$ where the $c$th class is the correct class.



In [ ]:
criterion = nn.CrossEntropyLoss() # cross-entropy loss
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001) # use Adam optimizer and set learning rate

In [ ]:
from utils import calculate_accuracy

# training loop

for epoch in range(5): # pass through the entire dataset a certain number of times (=number of epochs)
  model.train()

  batch_num = 1

  # loop through one pass of the entire dataset (64 images per iteration)
  for images, labels in train_loader:
    # push data (images and labels tensors) onto GPU
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad() # clear old gradients
    outputs = model(images) # forward pass
    loss = criterion(outputs, labels) # compute loss
    loss.backward() # calculate gradient of loss wrt weights using backpropagation
    optimizer.step() # update weights based on gradient

    # intermittently printing loss + accuracy on current batch
    if batch_num == 1 or batch_num % 200 == 0:
      print(f"epoch: {epoch + 1}  |  batch: {batch_num}  |  loss: {loss.item()}  |  accuracy: {calculate_accuracy(outputs, labels)} \n")

    batch_num += 1
  print(f"---------- epoch {epoch+1} complete -----------\n")





epoch: 1  |  batch: 1  |  loss: 2.303295612335205  |  accuracy: 4.6875 

epoch: 1  |  batch: 200  |  loss: 0.5471100807189941  |  accuracy: 78.125 

epoch: 1  |  batch: 400  |  loss: 0.4005795121192932  |  accuracy: 87.5 

epoch: 1  |  batch: 600  |  loss: 0.28261125087738037  |  accuracy: 93.75 

epoch: 1  |  batch: 800  |  loss: 0.5799232721328735  |  accuracy: 76.5625 

---------- epoch 1 complete -----------

epoch: 2  |  batch: 1  |  loss: 0.4213322401046753  |  accuracy: 84.375 

epoch: 2  |  batch: 200  |  loss: 0.20078304409980774  |  accuracy: 92.1875 

epoch: 2  |  batch: 400  |  loss: 0.29010432958602905  |  accuracy: 89.0625 

epoch: 2  |  batch: 600  |  loss: 0.2941064238548279  |  accuracy: 90.625 

epoch: 2  |  batch: 800  |  loss: 0.201205313205719  |  accuracy: 92.1875 

---------- epoch 2 complete -----------

epoch: 3  |  batch: 1  |  loss: 0.24584947526454926  |  accuracy: 89.0625 

epoch: 3  |  batch: 200  |  loss: 0.2139674872159958  |  accuracy: 93.75 

epoch: 3 

In [ ]:
import torch

# save weights in a file
MODEL_SAVE_PATH = "fashion_model_weights.pth"
torch.save(model.state_dict(), MODEL_SAVE_PATH)